# Multi-Instrument Separation with BS-RoFormer

This notebook separates a mixed music track into individual stems. By default, it uses **BS-RoFormer-SW**, the toolkit's recommended six-stem model: vocals, drums, bass, guitar, piano, and other (remaining sounds).

For more source categories, switch to the optional Mega 53-stem model in the configuration cell. That model needs a GPU with about 16 GB of VRAM; the toolkit documentation notes that individual stems may be weaker than those from the six-stem model.

> Internet access is required to install the toolkit and download the model the first time. A GPU is recommended on Kaggle or Colab. Place your audio at `input/input_music.mp3` or `input/input_music.wav`; edit the configuration cell to change the path. WAV files are used directly without reconversion.

## 1. Configure

Change `SOURCE_AUDIO` if your audio file is stored elsewhere. `DEVICE = "auto"` selects CUDA when a GPU is available; otherwise, it falls back to CPU, which is slower.

In [ ]:
from pathlib import Path

AUDIO_CANDIDATES = [Path("input/input_music.wav"), Path("input/input_music.mp3")]
SOURCE_AUDIO = next((path for path in AUDIO_CANDIDATES if path.is_file()), AUDIO_CANDIDATES[0])
MODEL_ID = "roformer-model-bs-roformer-sw-by-jarredou"
# Optional: use more stems (requires about 16 GB of VRAM):
# MODEL_ID = "roformer-model-bs-roformer-mvsep-mega-53-stems"
DEVICE = "auto"

MODEL_DIR = Path("models").resolve()
WAV_INPUT_DIR = Path("input_wav").resolve()
OUTPUT_DIR = Path("output/instruments").resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
WAV_INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Source audio:", SOURCE_AUDIO.resolve())
print("Model:", MODEL_ID)
print("Output directory:", OUTPUT_DIR)


## 2. Install FFmpeg

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

## 3. Install BS-RoFormer-Infer

Install from the official repository. The PyPI releases are historical snapshots; the maintained version is installed directly from GitHub.

In [ ]:
!pip install -q "bs-roformer-infer @ git+https://github.com/openmirlab/bs-roformer-infer.git"

## 4. Download the model

The default model is about 700 MB. Internet access is required for the first download; later runs can reuse the model files saved in `models`.

In [ ]:
import subprocess

subprocess.run(
    ["bs-roformer-download", "--list-models"],
    check=True,
)

subprocess.run(
    [
        "bs-roformer-download",
        "--model", MODEL_ID,
        "--output-dir", str(MODEL_DIR),
    ],
    check=True,
)

If the source file is already WAV, the notebook uses it directly and skips FFmpeg. Other formats are converted to stereo WAV at 44.1 kHz.


In [ ]:
if not SOURCE_AUDIO.is_file():
    raise FileNotFoundError(
        f"Cannot find {SOURCE_AUDIO}. "
        "Place the audio file in the input folder or update SOURCE_AUDIO."
    )

if SOURCE_AUDIO.suffix.lower() == ".wav":
    inference_input_dir = SOURCE_AUDIO.resolve().parent
    prepared_audio = SOURCE_AUDIO.resolve()
    print("The file is already WAV; skipping conversion:", prepared_audio)
else:
    WAV_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    prepared_audio = WAV_INPUT_DIR / f"{SOURCE_AUDIO.stem}.wav"
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(SOURCE_AUDIO),
            "-ar", "44100",
            "-ac", "2",
            str(prepared_audio),
        ],
        check=True,
    )
    inference_input_dir = WAV_INPUT_DIR
    print("Converted:", prepared_audio)


## 6. Separate the stems

Results are saved to `output/instruments`. With the default model, the stems are vocals, drums, bass, guitar, piano, and other. The toolkit also creates a combined instrumental stem.

In [ ]:
subprocess.run(
    [
        "bs-roformer-infer",
        "--model", MODEL_ID,
        "--models_dir", str(MODEL_DIR),
        "--input_folder", str(inference_input_dir),
        "--store_dir", str(OUTPUT_DIR),
        "--device", DEVICE,
    ],
    check=True,
)


## 7. View the output files

In [ ]:
stem_files = sorted(OUTPUT_DIR.glob("*.wav"))
for stem_file in stem_files:
    print(stem_file)

if not stem_files:
    print("No WAV files found in the output folder; check the inference log above.")
